# 04 — Temporal Split + Train Undersampling 1:50

Notebook ini mengganti split lama `80% train / 20% test` menjadi split yang lebih optimal untuk research:

- **70% train**: untuk melatih model
- **10% validation**: untuk memilih threshold, epoch terbaik, dan hyperparameter
- **20% test**: untuk evaluasi final sekali saja

Semua split dilakukan secara **temporal** berdasarkan `timestamp`, bukan random split. Undersampling hanya dilakukan pada **training set**, sedangkan validation dan test tetap menggunakan distribusi real-world.


In [1]:
import pandas as pd
from pathlib import Path

RANDOM_STATE = 42
MAX_RATIO = 50

DATA_DIR = Path("../data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_DIR / "nft_confidence_filtered.csv"

LABEL_COL = "label_final"
TIME_COL = "timestamp"

assert INPUT_PATH.exists(), f"File tidak ditemukan: {INPUT_PATH}"

df = pd.read_csv(INPUT_PATH)

assert LABEL_COL in df.columns, f"Kolom label tidak ditemukan: {LABEL_COL}"
assert TIME_COL in df.columns, f"Kolom timestamp tidak ditemukan: {TIME_COL}"

df[LABEL_COL] = df[LABEL_COL].astype(int)
df = df.sort_values(TIME_COL).reset_index(drop=True)

print("Full dataset shape:", df.shape)
print("Full dataset distribution:")
print(df[LABEL_COL].value_counts().sort_index())
print("Fraud rate:", df[LABEL_COL].mean())


C:\Users\VENTUS\AppData\Local\Temp\ipykernel_2272\1158164173.py:17: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_PATH)


Full dataset shape: (1490796, 25)
Full dataset distribution:
label_final
0    1489978
1        818
Name: count, dtype: int64
Fraud rate: 0.0005487001574997518


## 1. Temporal Split 70/10/20

Split dilakukan berdasarkan urutan waktu:

```text
70% data awal       -> train
10% data berikutnya -> validation
20% data terakhir   -> test
```

Validation diperlukan agar threshold tuning tidak dilakukan langsung pada test set.


In [4]:
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.80)

train_df_original = df.iloc[:train_end].copy().reset_index(drop=True)
val_df = df.iloc[train_end:val_end].copy().reset_index(drop=True)
test_df = df.iloc[val_end:].copy().reset_index(drop=True)

print("Train original:", train_df_original.shape)
print(train_df_original[LABEL_COL].value_counts().sort_index())
print("Fraud rate:", train_df_original[LABEL_COL].mean())

print("Validation:", val_df.shape)
print(val_df[LABEL_COL].value_counts().sort_index())
print("Fraud rate:", val_df[LABEL_COL].mean())

print("Test:", test_df.shape)
print(test_df[LABEL_COL].value_counts().sort_index())
print("Fraud rate:", test_df[LABEL_COL].mean())


Train original: (1043557, 25)
label_final
0    1042855
1        702
Name: count, dtype: int64
Fraud rate: 0.0006726992392365726
Validation: (149079, 25)
label_final
0    149027
1        52
Name: count, dtype: int64
Fraud rate: 0.0003488083499352692
Test: (298160, 25)
label_final
0    298096
1        64
Name: count, dtype: int64
Fraud rate: 0.00021464985242822645


## 2. Validasi Temporal Order

Cell ini memastikan tidak ada temporal leakage:

```text
max timestamp train <= min timestamp validation <= max timestamp validation <= min timestamp test
```


In [5]:
print("Train time range:", train_df_original[TIME_COL].min(), "->", train_df_original[TIME_COL].max())
print("Val time range:  ", val_df[TIME_COL].min(), "->", val_df[TIME_COL].max())
print("Test time range: ", test_df[TIME_COL].min(), "->", test_df[TIME_COL].max())

assert train_df_original[TIME_COL].max() <= val_df[TIME_COL].min(), "Temporal leakage: train overlap dengan validation"
assert val_df[TIME_COL].max() <= test_df[TIME_COL].min(), "Temporal leakage: validation overlap dengan test"

print("Temporal split valid: tidak ada leakage antar split.")


Train time range: 1622505626 -> 1629695273
Val time range:   1629695273 -> 1629978053
Test time range:  1629978053 -> 1630454395
Temporal split valid: tidak ada leakage antar split.


## 3. Undersampling Hanya pada Training Set

Validation dan test **tidak** di-undersampling agar tetap merepresentasikan distribusi real-world.

Training set di-undersampling ke rasio maksimum:

```text
fraud : normal = 1 : 50
```


In [6]:
fraud_train = train_df_original[train_df_original[LABEL_COL] == 1].copy()
normal_train = train_df_original[train_df_original[LABEL_COL] == 0].copy()

target_normal = min(len(normal_train), len(fraud_train) * MAX_RATIO)

print("Fraud train:", len(fraud_train))
print("Normal train before undersampling:", len(normal_train))
print("Target normal after undersampling:", target_normal)

assert len(fraud_train) > 0, "Tidak ada fraud di training set. Split perlu dievaluasi ulang."
assert target_normal > 0, "Target normal undersampling tidak valid."


Fraud train: 702
Normal train before undersampling: 1042855
Target normal after undersampling: 35100


In [7]:
normal_train_sampled = normal_train.sample(
    n=target_normal,
    random_state=RANDOM_STATE
)

train_1_50 = pd.concat([fraud_train, normal_train_sampled], axis=0)
train_1_50 = train_1_50.sort_values(TIME_COL).reset_index(drop=True)

print("Train after undersampling:", train_1_50.shape)
print(train_1_50[LABEL_COL].value_counts().sort_index())

fraud_count = (train_1_50[LABEL_COL] == 1).sum()
normal_count = (train_1_50[LABEL_COL] == 0).sum()

print("Final train fraud:", fraud_count)
print("Final train normal:", normal_count)
print("Final train ratio normal/fraud:", normal_count / fraud_count)


Train after undersampling: (35802, 25)
label_final
0    35100
1      702
Name: count, dtype: int64
Final train fraud: 702
Final train normal: 35100
Final train ratio normal/fraud: 50.0


## 4. Simpan Dataset Split Baru

File output utama:

- `train_1_50.csv` → training set setelah undersampling
- `train_temporal_original.csv` → training set original sebelum undersampling
- `val_temporal.csv` → validation set real-world distribution
- `test_temporal.csv` → test set real-world distribution

Model TGN sebaiknya memakai:

```text
train_1_50.csv      -> training
val_temporal.csv    -> threshold tuning / model selection
test_temporal.csv   -> final evaluation
```


In [8]:
train_1_50.to_csv(DATA_DIR / "train_1_50.csv", index=False)
train_df_original.to_csv(DATA_DIR / "train_temporal_original.csv", index=False)
val_df.to_csv(DATA_DIR / "val_temporal.csv", index=False)
test_df.to_csv(DATA_DIR / "test_temporal.csv", index=False)

print("Saved files:")
print(DATA_DIR / "train_1_50.csv")
print(DATA_DIR / "train_temporal_original.csv")
print(DATA_DIR / "val_temporal.csv")
print(DATA_DIR / "test_temporal.csv")


Saved files:
..\data\processed\train_1_50.csv
..\data\processed\train_temporal_original.csv
..\data\processed\val_temporal.csv
..\data\processed\test_temporal.csv


## 5. Ringkasan Metodologi untuk Laporan

Karena data transaksi NFT bersifat temporal dan sequential, dataset diurutkan berdasarkan `timestamp` sebelum dilakukan pemisahan data. Split dilakukan secara temporal menggunakan rasio 70% training, 10% validation, dan 20% testing untuk menghindari temporal leakage.

Undersampling hanya diterapkan pada training set dengan rasio maksimum 1:50 antara wash trading dan normal transaction. Validation dan testing tidak di-undersampling agar evaluasi tetap merepresentasikan distribusi real-world.

Validation set digunakan untuk memilih threshold, epoch terbaik, dan hyperparameter. Test set hanya digunakan untuk evaluasi final model.
